# 03 — Benchmark DM (T3)


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from src.dataset import OpenBanditDataset
from obp.ope import DirectMethod, OffPolicyEvaluation

sns.set_theme(style="whitegrid")
np.random.seed(42)

POLICY_RANDOM = "random"
POLICY_BTS = "bts"
CAMPAIGN = "all"
N_ACTIONS = 80
LEN_LIST = 3
N_CONTEXT_FEATURES = 30
RANDOM_STATE = 42

FIGURES_DIR = Path("../figures/week3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_SAMPLES = 200


In [ ]:
dataset_random = OpenBanditDataset(behavior_policy=POLICY_RANDOM, campaign=CAMPAIGN)
feedback_random = dataset_random.obtain_batch_bandit_feedback()

dataset_bts = OpenBanditDataset(behavior_policy=POLICY_BTS, campaign=CAMPAIGN)
feedback_bts = dataset_bts.obtain_batch_bandit_feedback()

print("random:", feedback_random["context"].shape, "n_rounds=", feedback_random["n_rounds"])
print("bts   :", feedback_bts["context"].shape, "n_rounds=", feedback_bts["n_rounds"])
print(f"naive CTR random={feedback_random['reward'].mean():.6f}")
print(f"naive CTR bts   ={feedback_bts['reward'].mean():.6f}")


In [ ]:
context_random = feedback_random["context"][:, :N_CONTEXT_FEATURES]
action_random = feedback_random["action"]
reward_random = feedback_random["reward"].astype(np.float32)

X = np.hstack([context_random, np.eye(N_ACTIONS)[action_random]])
y = reward_random

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(n_neg / max(n_pos, 1)),
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)
print("Model fitted.")


In [4]:
def expected_rewards_for_feedback(model, feedback, n_actions=N_ACTIONS):
    context_dim_train = int(model.n_features_in_ - n_actions)
    if feedback["context"].shape[1] < context_dim_train:
        raise ValueError(
            f"Feedback has too few context features: {feedback['context'].shape[1]} < {context_dim_train}"
        )

    context = feedback["context"][:, :context_dim_train]
    n_rounds = context.shape[0]
    est = np.zeros((n_rounds, n_actions), dtype=np.float32)

    eye_actions = np.eye(n_actions, dtype=np.float32)
    for a in range(n_actions):
        a_mat = np.tile(eye_actions[a], (n_rounds, 1))
        X_a = np.hstack([context, a_mat])
        est[:, a] = model.predict_proba(X_a)[:, 1]

    return est


def action_dist_logged(feedback, n_actions=N_ACTIONS, len_list=LEN_LIST):
    n_rounds = int(feedback["n_rounds"])
    positions = feedback["position"].astype(int)
    actions = feedback["action"].astype(int)
    idx = np.arange(n_rounds)

    ad = np.ones((n_rounds, n_actions, len_list), dtype=np.float64) / n_actions
    ad[idx, :, positions] = 0.0
    ad[idx, actions, positions] = 1.0
    return ad


def action_dist_uniform(n_rounds, n_actions=N_ACTIONS, len_list=LEN_LIST):
    return np.full((n_rounds, n_actions, len_list), 1.0 / n_actions, dtype=np.float64)


In [ ]:
expected_reward_bts = expected_rewards_for_feedback(model, feedback_bts)
expected_reward_3d = np.tile(expected_reward_bts[:, :, np.newaxis], (1, 1, LEN_LIST)).astype(np.float64)

n_rounds_bts = int(feedback_bts["n_rounds"])
action_dist_bts = action_dist_logged(feedback_bts)
action_dist_random = action_dist_uniform(n_rounds_bts)

dm = DirectMethod()
ope = OffPolicyEvaluation(bandit_feedback=feedback_bts, ope_estimators=[dm])

v_dm_bts = dm.estimate_policy_value(
    action_dist=action_dist_bts,
    estimated_rewards_by_reg_model=expected_reward_3d,
    position=feedback_bts["position"],
)
v_dm_random = dm.estimate_policy_value(
    action_dist=action_dist_random,
    estimated_rewards_by_reg_model=expected_reward_3d,
    position=feedback_bts["position"],
)

ci_bts = ope.estimate_intervals(
    action_dist=action_dist_bts,
    estimated_rewards_by_reg_model=expected_reward_3d,
    n_bootstrap_samples=BOOTSTRAP_SAMPLES,
    random_state=RANDOM_STATE,
)["dm"]

ci_random = ope.estimate_intervals(
    action_dist=action_dist_random,
    estimated_rewards_by_reg_model=expected_reward_3d,
    n_bootstrap_samples=BOOTSTRAP_SAMPLES,
    random_state=RANDOM_STATE,
)["dm"]

results_week3 = pd.DataFrame(
    {
        "policy": ["random_baseline", "bts_logged_style"],
        "naive_ctr": [feedback_random["reward"].mean(), feedback_bts["reward"].mean()],
        "V_DM": [v_dm_random, v_dm_bts],
        "CI_low": [ci_random["95.0% CI (lower)"], ci_bts["95.0% CI (lower)"]],
        "CI_high": [ci_random["95.0% CI (upper)"], ci_bts["95.0% CI (upper)"]],
    }
)

results_week3


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results_week3.copy()

x = np.arange(len(plot_df))
width = 0.35

bars_naive = ax.bar(x - width / 2, plot_df["naive_ctr"], width, label="Naive CTR", alpha=0.85)
bars_dm = ax.bar(x + width / 2, plot_df["V_DM"], width, label="Direct Method", alpha=0.85)

err_low = plot_df["V_DM"] - plot_df["CI_low"]
err_high = plot_df["CI_high"] - plot_df["V_DM"]
ax.errorbar(x + width / 2, plot_df["V_DM"], yerr=[err_low, err_high], fmt="none", capsize=5, color="black", lw=1.2)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["policy"])
ax.set_ylabel("Estimated policy value")
ax.set_title("Week 3: DM benchmark reproduction (with bootstrap 95% CI)")
ax.legend()

for b in bars_dm:
    h = b.get_height()
    ax.annotate(f"{h:.4f}", (b.get_x() + b.get_width() / 2, h), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=9)

plt.tight_layout()
out_path = FIGURES_DIR / "week3_dm_vs_naive_ci.png"
plt.savefig(out_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {out_path}")
